# FightWatch — fine-tune từ đúng RGB-aux đang dùng
Lượt này được phép fine-tune dài hơn nếu development tiến triển; giới hạn cứng toàn phiên 35 phút, không tự chạy lại. Lần bootstrap trước đã tự ngắt do Git quét file không theo dõi trên Drive; lượt sửa giảm ngân sách còn 35 phút, không cộng thêm 45 phút. Chưa tự bật GPU. Chỉ chạy sau khi notebook CPU đã có annotations.reviewed.json hợp lệ và đã thống nhất ngân sách GPU mới. Giữ nguyên classifier gốc trước training; tạo lại feature cache theo SHA checkpoint và cửa sổ đã duyệt. Không dùng hai video iPhone để train/chọn ngưỡng. Không tự cài model.

In [ ]:
import time, threading
from google.colab import runtime
import torch
assert torch.cuda.is_available(), 'Chỉ chọn GPU sau khi preflight nhãn và ngân sách được duyệt; notebook này không tự bật GPU.'
if 'GPU_DEADLINE' not in globals():
    GPU_STARTED=time.monotonic()
    GPU_DEADLINE=GPU_STARTED+35*60
    WATCHDOG=threading.Timer(45*60,runtime.unassign)
    WATCHDOG.daemon=True
    WATCHDOG.start()
print('Watchdog toàn phiên 35 phút, tính cả mount/setup/staging. Chạy lại cell không reset ngân sách.')


In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
from google.colab import drive
try:
    drive.mount('/content/drive',timeout_ms=180000)
    ROOT=Path('/content/drive/MyDrive/SENTINEL')
    REPO=ROOT/'repo/anomaly-behavior-detector-system'
    LOCAL=Path('/content/fightwatch-refinement-code')
    SOURCE=ROOT/'artifacts/fight_fusion_v1'
    REVIEW=ROOT/'artifacts/fight_temporal_review_v1'
    DATA=Path('/content/fightwatch-reviewed-public-data')
    subprocess.run(['git','-C',str(REPO),'restore','--','colab'],check=True,timeout=60)
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=no'],text=True,timeout=60)
    assert not dirty.strip(), dirty
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'],check=True,timeout=90)
    REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    LOCAL.mkdir(parents=True,exist_ok=True)
    snapshot=Path('/content/refinement-code.zip')
    with snapshot.open('wb') as handle:
        subprocess.run(['git','-C',str(REPO),'archive','--format=zip','HEAD'],stdout=handle,check=True,timeout=90)
    import zipfile
    with zipfile.ZipFile(snapshot) as archive:
        assert all(not Path(item.filename).is_absolute() and '..' not in Path(item.filename).parts for item in archive.infolist())
        archive.extractall(LOCAL)
    os.chdir(LOCAL)
    from scripts.colab_process import run_logged
    print('Code:',REVISION)
except BaseException:
    runtime.unassign()
    raise


In [ ]:
import zipfile
OUTPUT=ROOT/'artifacts/fight_refinement_v1'
ANNOTATIONS=REVIEW/'annotations.reviewed.json'
def remaining(reserve=0):
    value=GPU_DEADLINE-time.monotonic()-reserve
    assert value>0,'Hết ngân sách toàn phiên'
    return value
try:
    assert not (OUTPUT/'run.json').exists(),'Giữ run cũ, không tự retry'
    assert ANNOTATIONS.exists(),'Thiếu nhãn đoạn đã duyệt; không training'
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],OUTPUT/'setup.log',min(240,remaining(240)),revision=REVISION)
    run_logged([sys.executable,'-m','scripts.fight_temporal_review','check','--manifest',str(SOURCE/'source_manifest.json'),'--annotations',str(ANNOTATIONS)],OUTPUT/'readiness.log',min(60,remaining(240)),revision=REVISION)
    base=ROOT/'artifacts/fight_large_v1'
    assert (base/'manifest.json').exists(),'Không tự tạo lại split'
    run_logged([sys.executable,'-m','scripts.fight_public_data','--archives',str(ROOT/'datasets/fight'),'--data',str(DATA),'--output',str(base)],OUTPUT/'data.log',remaining(360),revision=REVISION)
    models=Path('/content/refinement-source-models')
    models.mkdir(exist_ok=True)
    with zipfile.ZipFile(SOURCE/'rgb_aux/sentinel_fight_only_runtime.zip') as archive:
        for name in ('fight_model.ts','fight_runtime.json','yolov8n-pose.pt'):
            (models/name).write_bytes(archive.read('models/'+name))
    from fight.refinement import load_deployed_base
    source_model,source_metadata,source_sha=load_deployed_base(models)
    print('Đúng classifier nguồn:',source_sha,'GPU:',torch.cuda.get_device_name(0),flush=True)
    budget=min(30,remaining(90)/60)
    assert budget>=7,'Không còn đủ thời gian cho đối chứng; không chạy một nửa rồi tuyên bố tốt hơn.'
    command=[sys.executable,'-m','scripts.fight_refinement_train','--manifest',str(SOURCE/'source_manifest.json'),'--annotations',str(ANNOTATIONS),'--models',str(models),'--data',str(DATA),'--cache','/content/refinement-window-cache','--output',str(OUTPUT),'--budget-minutes',str(budget),'--variant-minutes',str(min(12,(budget-6)/2))]
    (OUTPUT/'provenance.json').write_text(json.dumps({'revision':REVISION,'source_classifier_sha256':source_sha,'scope':'Reviewed public diagnostic; no private phone upload; no auto-deployment'},indent=2))
    shutil.copyfile(ANNOTATIONS,OUTPUT/'annotations_frozen.json')
    run_logged(command,OUTPUT/'training.log',remaining(60),revision=REVISION)
    report=json.loads((OUTPUT/'run.json').read_text())
    print('FINAL:',report['status'],'selection:',report.get('recommended_by_development_only'),flush=True)
    for name,item in report['variants'].items():
        print(name,'protected progress:',item['protected_progress'],'diagnostics:',item.get('test_by_dataset'),flush=True)
    print('Artifacts:',OUTPUT,'seconds allocation:',round(time.monotonic()-GPU_STARTED),flush=True)
finally:
    try:
        cache=Path('/content/refinement-window-cache')
        if cache.exists() and OUTPUT.exists():
            saved=OUTPUT/'reviewed_window_cache.partial.zip'
            with zipfile.ZipFile(saved,'w',compression=zipfile.ZIP_STORED) as archive:
                for path in sorted(cache.glob('*.npz')):
                    if not path.name.endswith('.partial.npz'):
                        archive.write(path,path.name)
            saved.replace(OUTPUT/'reviewed_window_cache.zip')
            print('Đã lưu cache public mới trên Drive.',flush=True)
    finally:
        print('Ngắt GPU; giữ artifact trên Drive. Chưa thay checkpoint local.',flush=True)
        runtime.unassign()
